# Obligatorio 1: Sistema solar

En esta práctica, se realizó un programa para generar una simulación del movimiento de los planetas del Sistema Solar utilizando el algoritmo de Verlet, así como el cálculo de invariantes como la energía, el momento lineal y el momento angular para comprobar que se mantienen constantes. Asimismo, se calculó el periodo de las órbitas de cada planeta.

## Herramientas
Para realizar la simulación, se utilizó C++; y para la generación de las gráficas, se programó en Python. Todo esto se uso en Visual Studio Code, instalado en una máquina virtual con el sistema operativo Linux Mint (Ubuntu). Como apoyo para la programación, se utilizó tanto el Copilot integrado en Visual Studio, como Gemini. En Visual Studio, el entorno que se utilizó fue uno virtual de Python, venv. El procesador del ordenador utilizado es un Intel Core i7-10750H CPU @ 2.60GHz; y la que se ha utilizado es Intel UHD Graphics, la integrada del sistema. 

## Datos
Para realizar la simulación, se consideraron todos los planetas del Sistema Solar y Plutón. Además, se consideró que, en el instante inicial, los planetas eran colineales en el eje X. Además, se utilizó la Constante de Gravitación Universal, con valor $G=6.67430·10^{-11}$ m³·s²/kg, la distancia de la Tierra al Sol con valor $c=1.496·10^{11}$ m y la masa del Sol, $M_{sol}=2·10^{30}$ kg. 

Los datos, almacenados de cada uno de los planetas en el fichero condiciones_iniciales, vienen dados en el siguiente formato: en primer lugar, se da la masa de cada planeta en kilogramos, a continuación, las componentes x e y de la posición inicial de cada planeta sobre el plano en metros y, por último, las componentes x e y de la velocidad orbital, que tomamos como inicial de cada planeta. Los datos fueron recopilados del documento Planetary Fact Sheet actualizado por la NASA a 18 de julio de 2018.

## Simulación de las órbitas.
El programa que realiza la simulación de las órbitas inicial, el cálculo de los invariantes (energía, momento lineal y momento angular) fue programado en C++. Sin embargo, el código necesitaba ser normalizado completamente a C++, puesto que se utilizaron funciones de C, que generaban errores. Dicho programa inicial es el siguiente:

```cpp
/***********ALGORITMO DE VERLET PARA SIMULACIÓN DEL SISTEMA SOLAR***********/

/*
Este programa tiene como objetivo ejecutar el algoritmo de Verlet para simular el movimiento de los planetas en el sistema solar. 
En primer lugar, se tomarán los datos de un fichero donde se colocarán las condiciones iniciales y datos sobre los planetas.
Los datos requeridos son la distancia entre el sol y el planeta, su masa y su velocidad inicial.
Una vez recopilados estos datos, a continuación se hace un reescalado para no trabajar con una combinación de números grandes
y pequeños, lo que podría generar problemas de precisión en los cálculos.
A continuación, se implementará el algoritmo de Verlet para calcular la posición y velocidad de cada planeta en cada paso de tiempo. 
Luego, se deshace el reescalado de los datos obtenidos.
Finalmente, se almacenarán en un nuevo fichero para, posteriormente, poder animar la trayectoria de los planetas utilizando Python.
*/


#include <iostream>
#include <math.h> // esto es C
#include <fstream>
#include <string>
#include <vector>

using namespace std;

// Dejo un hueco para definir funciones
//Función con el agoritmo de Verlet
void Verlet(double &t, double h, int N, double x0[][2], double v0[][2], double x[][4][2], double v[][4][2], double a[][4][2], double masa[]);
//Función para reescalar los datos de entrada
void reescalar(double &h, double x0[][2], double v0[][2], double masa[]);
//Función para deshacer el reescalado de los datos obtenidos
void deshacer_reescalado(double x[][4][2], double v[][4][2], double a[][4][2], double masa[], int N);
//Función para leer el fichero de datos de entrada
void leer_datos(ifstream &data, double x0[][2], double v0[][2], double masa[]);
//Función para escribir en el fichero de salida
void escribir_datos(ofstream &datafileout, double x[][4][2], int N);
void escribir_datos_energia(ofstream &datafileout, double E[][4], int N);
void escribir_datos_periodo(ofstream &datafileout, double periodo[]);
//Función para calcular los invariantes
void invariantes(double x[][4][2], double v[][4][2], double a[][4][2], double masa[], double E[][4], double L[][4][2], double p[][4][2], double mod_p[][4], int N);
//Función para calcular los periodos utilizando la tercera ley de Kepler
void periodos(double E[][4], double masa[], double periodos[], int N);

int main() {

    //Añado los ficheros
    ifstream data("condiciones_iniciales.txt");
    ofstream trayectoriasverlet("posiciones_planetas.dat");
    ofstream momento_angular("momento_angular.dat");
    ofstream energia("energia.dat");
    ofstream momento_lineal("momento_lineal.dat");
    ofstream periodo_file("periodos.dat");

    //Añado las variables del tiempo y el paso
    double t;
    double h;
    //Añado el número de pasos que queremos realizar. Esto es modificable según se necesite.
    int N;
    N=1000;

    //Añado el tiempo inicial y el paso. Esto tambien se debe modificar si es necesario.
    t=0.0;
    h=0.01;


    //Creo las matrices con las condiciones iniciales. El numero de planetas puede ser modificado si hace falta.
    double x0[4][2];    //Posiciones iniciales. x0[numero de planetas][coordenada]
    double v0[4][2];    //Velocidades iniciales. v0[numero de planetas][coordenada]

    //Creo matrices tridimensionales que almacenarán las posiciones, velocidades y aceleraciones de los planetas en cada paso de tiempo.
    double x[N][4][2];  //Posiciones. x[numero de pasos][numero de planetas][coordenada]
    double v[N][4][2];  //Velocidades. v[numero de pasos][numero de planetas][coordenada]
    double a[N][4][2];  //Aceleraciones. a[numero de pasos][numero de planetas][coordenada]

    //Necesito también un vector con las masas de cada planeta.
    double masa[4];

    //Preparo las matrices de los invariantes.
    double E[N][4]; //Energía total de cada planeta en cada paso de tiempo. E[numero de pasos][numero de planetas]
    double L[N][4][2]; //Momento angular total de cada planeta en cada paso de tiempo. L[numero de pasos][numero de planetas][componente]
    double p[N][4][2]; //Momento lineal de cada planeta en cada paso de tiempo. P[numero de pasos][numero de planetas][componente]
    double mod_p[N][4]; //Módulo del momento lineal de cada planeta en cada paso de tiempo. mod_p[numero de pasos][numero de planetas]

    //Preparo el vector de periodos de cada planeta.
    double periodo[4];

    //Leo los datos de entrada del fichero y los almaceno en las matrices correspondientes.
    leer_datos(data, x0, v0, masa);

    //Reescalo los datos de entrada para no trabajar con números muy grandes o muy pequeños.
    reescalar(h, x0, v0, masa);

    //Dado el tiempo inicial, el paso y el número de pasos, genero un vector de tiempos para cada paso.
    //Este vector de tiempo está afectado por el reescalado, que se aplicó al paso de tiempo. Más adelante se deshará el reescalado de este vector de tiempo.
    double tiempo[N];
    for (int i=0; i<N; i++){
        tiempo[i]=i*h;
    };

    //Aplico el algoritmo de Verlet para calcular la posición, velocidad y aceleración de cada planeta en cada paso de tiempo.
    Verlet(t, h, N, x0, v0, x, v, a, masa);

    //Escribo los datos de posición de cada planeta en cada paso de tiempo en el fichero de salida.
    //Lo hago antes de deshacer el reescalado para facilitar el dibujado de las trayectorias con Python.
    escribir_datos(trayectoriasverlet, x, N);

    //Deshago el reescalado de los datos obtenidos, para luego calcular los invariantes.
    deshacer_reescalado(x, v, a, masa, N);

    //Una vez hecho esto, la simulación de las trayectorias está completa.

    //Calculo la energía total, el momento angular total y el momento lineal para cada planeta y paso de tiempo.
    invariantes(x, v, a, masa, E, L, p, mod_p, N);
    //Asimismo, calculo los periodos.
    periodos(E, masa, periodo, N);

    //Escribo los resultados de energía, momento angular y momento lineal en los ficheros correspondientes, así como los periodos.
    escribir_datos(momento_angular, L, N);
    escribir_datos_energia(momento_lineal, mod_p, N);
    escribir_datos_energia(energia, E, N);
    escribir_datos_periodo(periodo_file, periodo);

    return 0;
}

//Aquí escribo el código de las funciones.
void Verlet(double &t, double h, int N, double x0[][2], double v0[][2], double x[][4][2], double v[][4][2], double a[][4][2], double masa[]) {

    //Esta función ignora los reescalados, por lo que se asume que los datos de entrada ya han sido reescalados
    //y se deshará el reescalamiento a posteriori.

    //Para implementar el algoritmo, necesito la constante de gravitación universal, que es la siguiente:
    double G=6.67430e-11;
    //También necesitaré un vector auxiliar de velocidad angular.
    double w[4][2]; //Velocidad angular de cada planeta en cada coordenada.
    //Me aseguro de que t=0 al inicio del algoritmo.
    t=0.0;

    //En primer lugar, se añaden los datos iniciales a las matrices de posiciones y velocidades.
    for (int i=0; i<4; i++){
        for (int j=0; j<2; j++){
            x[0][i][j]=x0[i][j];
            v[0][i][j]=v0[i][j];
        }
    };

    //A continuación comienzo con el algoritmo de Verlet.
    for (int n=0; n<N-1;n++){         //Itero para cada paso de tiempo
        for (int i=0; i<4; i++){    //Itero para cada planeta
            for(int j=0; j<2; j++){ //Itero para cada componente de posición, velocidad y aceleración.
                //En primer lugar, calculo la aceleración de cada planeta en el paso t.
                a[n][i][j]=0.0; //Inicializo la aceleración a cero.
                a[n+1][i][j]=0.0; //Inicializo la aceleración en el paso t+h a cero.
                for (int k = 0; k < 4; k++) {
                    if (k == i) continue; // Saltamos el propio planeta
    
                    // Calculamos la distancia (mejor fuera del bucle 'j' para ahorrar CPU)
                    double dx = x[n][i][0] - x[n][k][0];
                    double dy = x[n][i][1] - x[n][k][1];
                    double dist3 = pow(dx*dx + dy*dy, 1.5);
    
                    // Acumulamos la aceleración con +=
                    a[n][i][j] += -G * masa[k] * (x[n][i][j] - x[n][k][j]) / dist3;
}
                //Una vez tengo la aceleración, calculo la posición en el paso t+h, utilizando la fórmula de Verlet.
                x[n+1][i][j]=x[n][i][j]+v[n][i][j]*h+0.5*a[n][i][j]*pow(h,2);
                //También calculo una velocidad angular que necesitaré más adelante.
                w[i][j]=v[n][i][j]+0.5*a[n][i][j]*h;
                //Calculo la aceleración en el paso t+h, utilizando la posición que acabo de calcular.
                for (int k = 0; k < 4; k++) {
                    if (k == i) continue; // Saltamos el propio planeta
    
                    // Calculamos la distancia (mejor fuera del bucle 'j' para ahorrar CPU)
                    double dx = x[n+1][i][0] - x[n+1][k][0];
                    double dy = x[n+1][i][1] - x[n+1][k][1];
                    double dist3 = pow(dx*dx + dy*dy, 1.5);
    
                    // Acumulamos la aceleración con +=
                    a[n+1][i][j] += -G * masa[k] * (x[n+1][i][j] - x[n+1][k][j]) / dist3;
                };
                //Calculo la velocidad en el paso t+h, utilizando la fórmula de Verlet.
                v[n+1][i][j]=w[i][j]+0.5*a[n+1][i][j]*h;
                //Por último, actualizo el tiempo.
                t+=h;
            }
        };

    };
    //Con esto ya tengo actualizados los vectores de datos de posición, velocidad y aceleración de cada planeta en cada paso de tiempo.
    return;
    
};

void leer_datos(ifstream &data, double x0[][2], double v0[][2], double masa[]) {
    //Esta función lee los datos de entrada del fichero "condiciones_iniciales.txt" y los almacena en las matrices correspondientes.
    //El formato del fichero es el siguiente:
    //Planeta 1: masa, posición x, posición y, velocidad x, velocidad y
    //Planeta 2: masa, posición x, posición y, velocidad x, velocidad y
    //Planeta 3: masa, posición x, posición y, velocidad x, velocidad y
    //Planeta 4: masa, posición x, posición y, velocidad x, velocidad y

    for (int i=0; i<4; i++){
        data >> masa[i] >> x0[i][0] >> x0[i][1] >> v0[i][0] >> v0[i][1];
    };

    return;
};

void escribir_datos(ofstream &datafileout, double x[][4][2], int N) {
    //Esta función escribe los datos de posición de cada planeta en cada paso de tiempo en el fichero "posiciones_planetas.dat".
    //El formato del fichero es el siguiente:
    //Paso de tiempo 1: posición x del planeta 1, posición y del planeta 1 [salto de línea] posición x del planeta 2, posición y del planeta 2, ...
    int i;
    int j;

    for (i=0; i<N; i++)
    {
        for(j=0; j<4; j++){
            datafileout << x[i][j][0] << ", " << x[i][j][1] << endl;
        }
    };

    return;
};

void reescalar(double &h, double x0[][2], double v0[][2], double masa[]){
    double c;
    double G;
    double M;

    c=1.496e11;
    G=6.67430e-11;
    M=2e30;
    //Reescalo todas las posiciones, velocidades y masas.
    for (int i=0; i<4; i++){
        x0[i][0]=x0[i][0]/c;
        x0[i][1]=x0[i][1]/c;
        v0[i][0]=v0[i][0]*pow(c,1.5)/(c*pow(G*M,0.5));
        v0[i][1]=v0[i][1]*pow(c,1.5)/(c*pow(G*M,0.5));
        masa[i]=masa[i]/M;
    };
    h=h*pow(G*M/pow(c,3),0.5);

};

void deshacer_reescalado(double x[][4][2], double v[][4][2], double a[][4][2], double masa[], int N){
    double c;
    double G;
    double M;

    c=1.496e11;
    G=6.67430e-11;
    M=2e30;
    //Deshago el reescalado de todas las posiciones, velocidades, aceleraciones y masas.
    for (int n=0; n<N; n++){
        for (int i=0; i<4; i++){
            for (int j=0; j<2; j++){
                x[n][i][j]=x[n][i][j]*c;
                v[n][i][j]=v[n][i][j]*c*pow(G*M/pow(c,3),-0.5);
                a[n][i][j]=a[n][i][j]*G*M/pow(c,2);
            };
            masa[i]=masa[i]*M;
        }
    };
    return;
};

void invariantes(double x[][4][2], double v[][4][2], double a[][4][2], double masa[], double E[][4], double L[][4][2], double p[][4][2], double mod_p[][4], int N){
    //Esta función calcula la energía total, el momento angular total y el momento lineal para cada planeta y paso de tiempo.
    //La energía total se calcula como la suma de la energía cinética y la energía potencial.
    //El momento angular total se calcula como el producto vectorial entre el vector de posición y el vector de velocidad.
    //El momento lineal se calcula como el producto entre la masa y la velocidad.
    double G=6.67430e-11;

    for (int n=0; n<N; n++){
        for (int i=0; i<4; i++){
            //Calculo la energía cinética.
            double energia_cinetica=0.0;
            for (int j=0; j<2; j++){
                energia_cinetica+=0.5*masa[i]*pow(v[n][i][j],2);
            };
            //Calculo la energía potencial.
            double energia_potencial=0.0;
            for (int k=0; k<4; k++){
                if (k==i) continue;
                double dx=x[n][i][0]-x[n][k][0];
                double dy=x[n][i][1]-x[n][k][1];
                double dist=sqrt(pow(dx,2)+pow(dy,2));
                energia_potencial+=-G*masa[i]*masa[k]/dist;
            };
            //Calculo la energía total.
            E[n][i]=energia_cinetica+energia_potencial;

            //Calculo el momento angular total.
            L[n][i][0]=masa[i]*(x[n][i][1]*v[n][i][0]-x[n][i][0]*v[n][i][1]);
            L[n][i][1]=masa[i]*(x[n][i][0]*v[n][i][1]-x[n][i][1]*v[n][i][0]);

            //Calculo el momento lineal y su módulo.
            p[n][i][0]=masa[i]*v[n][i][0];
            p[n][i][1]=masa[i]*v[n][i][1];

            mod_p[n][i]=sqrt(pow(p[n][i][0],2)+pow(p[n][i][1],2));
        }
    };


    return;
};

void escribir_datos_energia(ofstream &datafileout, double E[][4], int N) {
    //Esta función escribe los datos de energía total de cada planeta en cada paso de tiempo en el fichero "energia.dat".
    //El formato del fichero es el siguiente:
    //Paso de tiempo 1: energía total del planeta 1, energía total del planeta 2, energía total del planeta 3, energía total del planeta 4 [salto de línea]
    int i;
    int j;

    for (i=0; i<N; i++)
    {
        for(j=0; j<4; j++){
            datafileout << E[i][j] << "\n";
        }
        datafileout << endl;
    };

    return;
};

void periodos(double E[][4], double masa[], double periodos[], int N){
    //Esta función calcula los periodos de cada planeta utilizando la tercera ley de Kepler.
    //La tercera ley de Kepler establece que el periodo de un planeta es proporcional a la raíz cúbica del semieje mayor de su órbita.
    //El semieje mayor se puede calcular a partir de la energía total del planeta utilizando la fórmula: a = -G*M/(2*E), donde G es la constante de gravitación universal, M es la masa del sol y E es la energía total del planeta.
    double G=6.67430e-11;
    double M=2e30;
    double energiasmedias[4];

    //Calculo la media de la energía total de cada planeta a lo largo de los pasos de tiempo.
    for (int i=0; i<4; i++){
        double suma_energias=0.0;
        for (int n=0; n<N; n++){
            suma_energias+=E[n][i];
        };
        energiasmedias[i]=suma_energias/N;
    };
    //Calculo el periodo usando la tercera ley de Kepler.
    for (int i=0; i<4; i++){
        double a=-G*M*masa[i]/(2*energiasmedias[i]);
        periodos[i]=2*M_PI*pow(a,1.5)/pow(G*M,0.5);
    };
       

    return;
};

void escribir_datos_periodo(ofstream &datafileout, double periodo[]){
    //Esta función escribe los datos de periodo de cada planeta en el fichero "periodos.dat".
    //El formato del fichero es el siguiente:
    //Planeta 1: periodo del planeta 1 [salto de línea]
    //Planeta 2: periodo del planeta 2 [salto de línea]
    //Planeta 3: periodo del planeta 3 [salto de línea]
    //Planeta 4: periodo del planeta 4 [salto de línea]

    for (int i=0; i<4; i++){
        datafileout << periodo[i] << "\n";
    };

    return;
}
```

La segunda versión del programa, que se presentará a continuación, fue nuevamente corregido con Copilot para añadir using namespace std. El código de la segunda versión es el siguiente:

```cpp

#include <array>
#include <cmath>
#include <fstream>
#include <iostream>
#include <vector>

using namespace std;

constexpr size_t kNumPlanetas = 4;
constexpr size_t kNumCoordenadas = 2;
constexpr double kG = 6.67430e-11;
constexpr double kUnidadDistancia = 1.496e11;
constexpr double kUnidadMasa = 2.0e30;

using Vector2D = array<double, kNumCoordenadas>;
using PlanetArray = array<Vector2D, kNumPlanetas>;
using Trajectory = vector<PlanetArray>;
using EnergyArray = vector<array<double, kNumPlanetas>>;
using MassArray = array<double, kNumPlanetas>;

void leer_datos(ifstream& data, PlanetArray& x0, PlanetArray& v0, MassArray& masa);
void reescalar(double& h, PlanetArray& x0, PlanetArray& v0, MassArray& masa);
void deshacer_reescalado(Trajectory& x, Trajectory& v, Trajectory& a, MassArray& masa);
void Verlet(double& t, double h, int N, const PlanetArray& x0, const PlanetArray& v0, Trajectory& x, Trajectory& v, Trajectory& a, const MassArray& masa);
PlanetArray calcular_aceleraciones(const PlanetArray& posiciones, const MassArray& masa);
void escribir_datos(ofstream& out, const Trajectory& x);
void escribir_datos_energia(ofstream& out, const EnergyArray& E);
void escribir_datos_periodo(ofstream& out, const array<double, kNumPlanetas>& periodos);
void invariantes(const Trajectory& x, const Trajectory& v, const Trajectory& a, const MassArray& masa, EnergyArray& E, Trajectory& L, Trajectory& p);
void periodos(const EnergyArray& E, const MassArray& masa, array<double, kNumPlanetas>& periodos);

int main() {
    ifstream data("condiciones_iniciales.txt");
    if (!data) {
        cerr << "Error: no se pudo abrir condiciones_iniciales.txt\n";
        return 1;
    }

    ofstream trayectorias("posiciones_planetas.dat");
    if (!trayectorias) {
        cerr << "Error: no se pudo crear posiciones_planetas.dat\n";
        return 1;
    }

    ofstream momento_angular("momento_angular.dat");
    if (!momento_angular) {
        cerr << "Error: no se pudo crear momento_angular.dat\n";
        return 1;
    }

    ofstream energia("energia.dat");
    if (!energia) {
        cerr << "Error: no se pudo crear energia.dat\n";
        return 1;
    }

    ofstream momento_lineal("momento_lineal.dat");
    if (!momento_lineal) {
        cerr << "Error: no se pudo crear momento_lineal.dat\n";
        return 1;
    }

    ofstream periodo_file("periodos.dat");
    if (!periodo_file) {
        cerr << "Error: no se pudo crear periodos.dat\n";
        return 1;
    }

    constexpr int N = 1000;
    double t = 0.0;
    double h = 0.01;

    PlanetArray x0{};
    PlanetArray v0{};
    MassArray masa{};

    leer_datos(data, x0, v0, masa);
    reescalar(h, x0, v0, masa);

    vector<double> tiempo(N);
    for (int i = 0; i < N; ++i) {
        tiempo[i] = i * h;
    }

    Trajectory x(N, PlanetArray{});
    Trajectory v(N, PlanetArray{});
    Trajectory a(N, PlanetArray{});
    EnergyArray E(N);
    Trajectory L(N, PlanetArray{});
    Trajectory p(N, PlanetArray{});
    array<double, kNumPlanetas> periodo{};

    Verlet(t, h, N, x0, v0, x, v, a, masa);
    escribir_datos(trayectorias, x);

    deshacer_reescalado(x, v, a, masa);
    invariantes(x, v, a, masa, E, L, p);
    periodos(E, masa, periodo);

    escribir_datos(momento_angular, L);
    escribir_datos(momento_lineal, p);
    escribir_datos_energia(energia, E);
    escribir_datos_periodo(periodo_file, periodo);

    return 0;
}

void leer_datos(ifstream& data, PlanetArray& x0, PlanetArray& v0, MassArray& masa) {
    for (size_t i = 0; i < kNumPlanetas; ++i) {
        data >> masa[i] >> x0[i][0] >> x0[i][1] >> v0[i][0] >> v0[i][1];
    }
}

void reescalar(double& h, PlanetArray& x0, PlanetArray& v0, MassArray& masa) {
    for (size_t i = 0; i < kNumPlanetas; ++i) {
        x0[i][0] /= kUnidadDistancia;
        x0[i][1] /= kUnidadDistancia;
        v0[i][0] *= pow(kUnidadDistancia, 1.5) / (kUnidadDistancia * sqrt(kG * kUnidadMasa));
        v0[i][1] *= pow(kUnidadDistancia, 1.5) / (kUnidadDistancia * sqrt(kG * kUnidadMasa));
        masa[i] /= kUnidadMasa;
    }

    h *= sqrt(kG * kUnidadMasa / (kUnidadDistancia * kUnidadDistancia * kUnidadDistancia));
}

void deshacer_reescalado(Trajectory& x, Trajectory& v, Trajectory& a, MassArray& masa) {
    const double factorVel = 1.0 / (pow(kUnidadDistancia, 1.5) / (kUnidadDistancia * sqrt(kG * kUnidadMasa)));
    const double factorAcel = kG * kUnidadMasa / (kUnidadDistancia * kUnidadDistancia);

    for (auto& paso : x) {
        for (auto& planeta : paso) {
            for (auto& componente : planeta) {
                componente *= kUnidadDistancia;
            }
        }
    }

    for (auto& paso : v) {
        for (auto& planeta : paso) {
            for (auto& componente : planeta) {
                componente *= factorVel;
            }
        }
    }

    for (auto& paso : a) {
        for (auto& planeta : paso) {
            for (auto& componente : planeta) {
                componente *= factorAcel;
            }
        }
    }

    for (auto& masa_planeta : masa) {
        masa_planeta *= kUnidadMasa;
    }
}

PlanetArray calcular_aceleraciones(const PlanetArray& posiciones, const MassArray& masa) {
    PlanetArray aceleraciones{};

    for (size_t i = 0; i < kNumPlanetas; ++i) {
        aceleraciones[i] = {0.0, 0.0};
        for (size_t k = 0; k < kNumPlanetas; ++k) {
            if (k == i) {
                continue;
            }

            const double dx = posiciones[i][0] - posiciones[k][0];
            const double dy = posiciones[i][1] - posiciones[k][1];
            const double distanciaCubica = pow(dx * dx + dy * dy, 1.5);

            aceleraciones[i][0] += -kG * masa[k] * dx / distanciaCubica;
            aceleraciones[i][1] += -kG * masa[k] * dy / distanciaCubica;
        }
    }

    return aceleraciones;
}

void Verlet(double& t, double h, int N, const PlanetArray& x0, const PlanetArray& v0, Trajectory& x, Trajectory& v, Trajectory& a, const MassArray& masa) {
    t = 0.0;
    x[0] = x0;
    v[0] = v0;
    a[0] = calcular_aceleraciones(x[0], masa);

    for (int n = 0; n + 1 < N; ++n) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            for (size_t j = 0; j < kNumCoordenadas; ++j) {
                x[n + 1][i][j] = x[n][i][j] + v[n][i][j] * h + 0.5 * a[n][i][j] * h * h;
            }
        }

        a[n + 1] = calcular_aceleraciones(x[n + 1], masa);

        for (size_t i = 0; i < kNumPlanetas; ++i) {
            for (size_t j = 0; j < kNumCoordenadas; ++j) {
                v[n + 1][i][j] = v[n][i][j] + 0.5 * (a[n][i][j] + a[n + 1][i][j]) * h;
            }
        }

        t += h;
    }
}

void escribir_datos(ofstream& out, const Trajectory& x) {
    for (const auto& paso : x) {
        for (const auto& planeta : paso) {
            out << planeta[0] << ", " << planeta[1] << '\n';
        }
        out << '\n';
    }
}

void escribir_datos_energia(ofstream& out, const EnergyArray& E) {
    for (const auto& paso : E) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            out << paso[i];
            if (i + 1 < kNumPlanetas) {
                out << ", ";
            }
        }
        out << '\n';
    }
}

void escribir_datos_periodo(ofstream& out, const array<double, kNumPlanetas>& periodos) {
    for (size_t i = 0; i < kNumPlanetas; ++i) {
        out << periodos[i] << '\n';
    }
}

void invariantes(const Trajectory& x, const Trajectory& v, const Trajectory& a, const MassArray& masa, EnergyArray& E, Trajectory& L, Trajectory& p) {
    for (size_t n = 0; n < x.size(); ++n) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            const double energia_cinetica = 0.5 * masa[i] * (v[n][i][0] * v[n][i][0] + v[n][i][1] * v[n][i][1]);
            double energia_potencial = 0.0;

            for (size_t k = 0; k < kNumPlanetas; ++k) {
                if (k == i) {
                    continue;
                }

                const double dx = x[n][i][0] - x[n][k][0];
                const double dy = x[n][i][1] - x[n][k][1];
                const double distancia = sqrt(dx * dx + dy * dy);
                energia_potencial += -kG * masa[i] * masa[k] / distancia;
            }

            E[n][i] = energia_cinetica + energia_potencial;
            L[n][i][0] = masa[i] * (x[n][i][1] * v[n][i][0] - x[n][i][0] * v[n][i][1]);
            L[n][i][1] = masa[i] * (x[n][i][0] * v[n][i][1] - x[n][i][1] * v[n][i][0]);
            p[n][i][0] = masa[i] * v[n][i][0];
            p[n][i][1] = masa[i] * v[n][i][1];
        }
    }
}

void periodos(const EnergyArray& E, const MassArray& masa, array<double, kNumPlanetas>& periodos) {
    array<double, kNumPlanetas> energia_media{};

    for (const auto& paso : E) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            energia_media[i] += paso[i];
        }
    }

    for (size_t i = 0; i < kNumPlanetas; ++i) {
        energia_media[i] /= static_cast<double>(E.size());
        const double semieje_mayor = -kG * kUnidadMasa * masa[i] / (2.0 * energia_media[i]);
        periodos[i] = 2.0 * M_PI * pow(semieje_mayor, 1.5) / sqrt(kG * kUnidadMasa);
    }
}

```

Finalmente, tras las revisiones, el código final, que fue ligeramente modificado para realizar unos ajustes que corregían ciertos errores, es el siguiente:

```cpp
#include <array>
#include <cmath>
#include <fstream>
#include <iostream>
#include <vector>

using namespace std;

constexpr size_t kNumPlanetas = 9;
constexpr size_t kNumCoordenadas = 2;
constexpr double kG = 6.67430e-11;
constexpr double kUnidadDistancia = 1.496e11;
constexpr double kUnidadMasa = 2.0e30;

using Vector2D = array<double, kNumCoordenadas>;
using PlanetArray = array<Vector2D, kNumPlanetas>;
using Trajectory = vector<PlanetArray>;
using EnergyArray = vector<array<double, kNumPlanetas>>;
using MassArray = array<double, kNumPlanetas>;

void leer_datos(ifstream& data, PlanetArray& x0, PlanetArray& v0, MassArray& masa);
void reescalar(double& h, PlanetArray& x0, PlanetArray& v0, MassArray& masa);
void deshacer_reescalado(Trajectory& x, Trajectory& v, Trajectory& a, MassArray& masa);
void Verlet(double& t, double h, int N, const PlanetArray& x0, const PlanetArray& v0, Trajectory& x, Trajectory& v, Trajectory& a, const MassArray& masa);
PlanetArray calcular_aceleraciones(const PlanetArray& posiciones, const MassArray& masa);
void escribir_datos(ofstream& out, const Trajectory& x);
void escribir_datos_energia(ofstream& out, const EnergyArray& data);
void escribir_datos_periodo(ofstream& out, const array<double, kNumPlanetas>& periodos);
void invariantes(const Trajectory& x, const Trajectory& v, const Trajectory& a, const MassArray& masa, EnergyArray& E, Trajectory& L, Trajectory& p, EnergyArray& mod_p);
void periodos(const EnergyArray& E, const MassArray& masa, array<double, kNumPlanetas>& periodos);
void convertir_periodo_a_dias(array<double, kNumPlanetas>& periodos);
int main() {
    ifstream data("condiciones_iniciales.txt");
    if (!data) {
        cerr << "Error: no se pudo abrir condiciones_iniciales.txt\n";
        return 1;
    }

    ofstream trayectorias("posiciones_planetas.dat");
    if (!trayectorias) {
        cerr << "Error: no se pudo crear posiciones_planetas.dat\n";
        return 1;
    }

    ofstream velocidades("velocidades_planetas.dat");
    if (!velocidades) {
        cerr << "Error: no se pudo crear velocidades_planetas.dat\n";
        return 1;
    }

    ofstream aceleraciones("aceleraciones_planetas.dat");
    if (!aceleraciones) {
        cerr << "Error: no se pudo crear aceleraciones_planetas.dat\n";
        return 1;
    }

    ofstream momento_angular("momento_angular.dat");
    if (!momento_angular) {
        cerr << "Error: no se pudo crear momento_angular.dat\n";
        return 1;
    }

    ofstream energia("energia.dat");
    if (!energia) {
        cerr << "Error: no se pudo crear energia.dat\n";
        return 1;
    }

    ofstream momento_lineal("momento_lineal.dat");
    if (!momento_lineal) {
        cerr << "Error: no se pudo crear momento_lineal.dat\n";
        return 1;
    }

    ofstream periodo_file("periodos.dat");
    if (!periodo_file) {
        cerr << "Error: no se pudo crear periodos.dat\n";
        return 1;
    }

    constexpr int N = 1500;
    double t = 0.0;
    double h = 86400;

    PlanetArray x0{};
    PlanetArray v0{};
    MassArray masa{};

    leer_datos(data, x0, v0, masa);
    reescalar(h, x0, v0, masa);

    vector<double> tiempo(N);
    for (int i = 0; i < N; ++i) {
        tiempo[i] = i * h;
    }

    Trajectory x(N, PlanetArray{});
    Trajectory v(N, PlanetArray{});
    Trajectory a(N, PlanetArray{});
    EnergyArray E(N);
    Trajectory L(N, PlanetArray{});
    Trajectory p(N, PlanetArray{});
    EnergyArray mod_p(N);
    array<double, kNumPlanetas> periodo{};

    Verlet(t, h, N, x0, v0, x, v, a, masa);
    escribir_datos(trayectorias, x);
    escribir_datos(velocidades, v);
    escribir_datos(aceleraciones, a);

    deshacer_reescalado(x, v, a, masa);
    invariantes(x, v, a, masa, E, L, p, mod_p);
    periodos(E, masa, periodo);
    convertir_periodo_a_dias(periodo);

    escribir_datos(momento_angular, L);
    escribir_datos_energia(momento_lineal, mod_p);
    escribir_datos_energia(energia, E);
    escribir_datos_periodo(periodo_file, periodo);

    return 0;
}

void leer_datos(ifstream& data, PlanetArray& x0, PlanetArray& v0, MassArray& masa) {
    for (size_t i = 0; i < kNumPlanetas; ++i) {
        data >> masa[i] >> x0[i][0] >> x0[i][1] >> v0[i][0] >> v0[i][1];
    }
}

void reescalar(double& h, PlanetArray& x0, PlanetArray& v0, MassArray& masa) {
    for (size_t i = 0; i < kNumPlanetas; ++i) {
        x0[i][0] /= kUnidadDistancia;
        x0[i][1] /= kUnidadDistancia;
        v0[i][0] *= pow(kUnidadDistancia, 1.5) / (kUnidadDistancia * sqrt(kG * kUnidadMasa));
        v0[i][1] *= pow(kUnidadDistancia, 1.5) / (kUnidadDistancia * sqrt(kG * kUnidadMasa));
        masa[i] /= kUnidadMasa;
    }

    h *= sqrt(kG * kUnidadMasa / (kUnidadDistancia * kUnidadDistancia * kUnidadDistancia));
}

void deshacer_reescalado(Trajectory& x, Trajectory& v, Trajectory& a, MassArray& masa) {
    const double factorVel = 1.0 / (pow(kUnidadDistancia, 1.5) / (kUnidadDistancia * sqrt(kG * kUnidadMasa)));
    const double factorAcel = kG * kUnidadMasa / (kUnidadDistancia * kUnidadDistancia);

    for (auto& paso : x) {
        for (auto& planeta : paso) {
            for (auto& componente : planeta) {
                componente *= kUnidadDistancia;
            }
        }
    }

    for (auto& paso : v) {
        for (auto& planeta : paso) {
            for (auto& componente : planeta) {
                componente *= factorVel;
            }
        }
    }

    for (auto& paso : a) {
        for (auto& planeta : paso) {
            for (auto& componente : planeta) {
                componente *= factorAcel;
            }
        }
    }

    for (auto& masa_planeta : masa) {
        masa_planeta *= kUnidadMasa;
    }
}

PlanetArray calcular_aceleraciones(const PlanetArray& posiciones, const MassArray& masa) {
    PlanetArray aceleraciones{};
    
    // Masa del sol en unidades escaladas
    constexpr double masa_sol = 1.989e30 / 2.0e30;

    for (size_t i = 0; i < kNumPlanetas; ++i) {
        aceleraciones[i] = {0.0, 0.0};
        
        // Contribución del sol (ubicado en el origen)
        const double dx_sol = posiciones[i][0];
        const double dy_sol = posiciones[i][1];
        const double dist_sol_cubica = pow(dx_sol * dx_sol + dy_sol * dy_sol, 1.5);
        
        aceleraciones[i][0] += -masa_sol * dx_sol / dist_sol_cubica;
        aceleraciones[i][1] += -masa_sol * dy_sol / dist_sol_cubica;
        
        // Contribución de los otros planetas
        for (size_t k = 0; k < kNumPlanetas; ++k) {
            if (k == i) {
                continue;
            }

            const double dx = posiciones[i][0] - posiciones[k][0];
            const double dy = posiciones[i][1] - posiciones[k][1];
            const double distanciaCubica = pow(dx * dx + dy * dy, 1.5);

            aceleraciones[i][0] += -masa[k] * dx / distanciaCubica;
            aceleraciones[i][1] += -masa[k] * dy / distanciaCubica;
        }
    }

    return aceleraciones;
}

void Verlet(double& t, double h, int N, const PlanetArray& x0, const PlanetArray& v0, Trajectory& x, Trajectory& v, Trajectory& a, const MassArray& masa) {
    t = 0.0;
    x[0] = x0;
    v[0] = v0;
    a[0] = calcular_aceleraciones(x[0], masa);

    for (int n = 0; n + 1 < N; ++n) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            for (size_t j = 0; j < kNumCoordenadas; ++j) {
                x[n + 1][i][j] = x[n][i][j] + v[n][i][j] * h + 0.5 * a[n][i][j] * h * h;
            }
        }

        a[n + 1] = calcular_aceleraciones(x[n + 1], masa);

        for (size_t i = 0; i < kNumPlanetas; ++i) {
            for (size_t j = 0; j < kNumCoordenadas; ++j) {
                v[n + 1][i][j] = v[n][i][j] + 0.5 * (a[n][i][j] + a[n + 1][i][j]) * h;
            }
        }

        t += h;
    }
}

void escribir_datos(ofstream& out, const Trajectory& x) {
    for (const auto& paso : x) {
        for (const auto& planeta : paso) {
            out << planeta[0] << ", " << planeta[1] << '\n';
        }
        out << '\n';
    }
}

void escribir_datos_energia(ofstream& out, const EnergyArray& data) {
    for (const auto& paso : data) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            out << paso[i] << '\n';
        }
        out << '\n';
    }
}

void escribir_datos_periodo(ofstream& out, const array<double, kNumPlanetas>& periodos) {
    for (size_t i = 0; i < kNumPlanetas; ++i) {
        out << periodos[i] << '\n';
    }
}

void invariantes(const Trajectory& x, const Trajectory& v, const Trajectory& a, const MassArray& masa, EnergyArray& E, Trajectory& L, Trajectory& p, EnergyArray& mod_p) {
    for (size_t n = 0; n < x.size(); ++n) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            const double energia_cinetica = 0.5 * masa[i] * (v[n][i][0] * v[n][i][0] + v[n][i][1] * v[n][i][1]);
            // ✅ Inicializamos la energía potencial con la del Sol
            double masa_sol = 2e30; // Masa del sol
            double dist_sol = sqrt(x[n][i][0] * x[n][i][0] + x[n][i][1] * x[n][i][1]);
            double energia_potencial = -kG * masa_sol * masa[i] / dist_sol;

            for (size_t k = 0; k < kNumPlanetas; ++k) {
                if (k == i) {
                    continue;
                }

                const double dx = x[n][i][0] - x[n][k][0];
                const double dy = x[n][i][1] - x[n][k][1];
                const double distancia = sqrt(dx * dx + dy * dy);
                energia_potencial += -kG * masa[i] * masa[k] / distancia;
            }

            E[n][i] = energia_cinetica + energia_potencial;
            L[n][i][0] = masa[i] * (x[n][i][1] * v[n][i][0] - x[n][i][0] * v[n][i][1]);
            L[n][i][1] = masa[i] * (x[n][i][0] * v[n][i][1] - x[n][i][1] * v[n][i][0]);
            p[n][i][0] = masa[i] * v[n][i][0];
            p[n][i][1] = masa[i] * v[n][i][1];
            mod_p[n][i] = sqrt(p[n][i][0] * p[n][i][0] + p[n][i][1] * p[n][i][1]);
        }
    }
}

void periodos(const EnergyArray& E, const MassArray& masa, array<double, kNumPlanetas>& periodos) {
    array<double, kNumPlanetas> energia_media{};

    for (const auto& paso : E) {
        for (size_t i = 0; i < kNumPlanetas; ++i) {
            energia_media[i] += paso[i];
        }
    }

    for (size_t i = 0; i < kNumPlanetas; ++i) {
        energia_media[i] /= static_cast<double>(E.size());
        const double semieje_mayor = -kG * kUnidadMasa * masa[i] / (2.0 * energia_media[i]);
        periodos[i] = 2.0 * M_PI * pow(semieje_mayor, 1.5) / sqrt(kG * kUnidadMasa);
    }
}

void convertir_periodo_a_dias(array<double, kNumPlanetas>& periodos) {
    constexpr double segundos_por_dia = 86400.0;
    for (auto& periodo : periodos) {
        periodo /= segundos_por_dia;
    }
}
```

Este programa está preparado para poder cambiar el número de planetas e iteraciones modificando el código directamente, así como el número de iteraciones y el tamaño del paso. Para realizar esta simulación, se trabajó con un paso de un día y un total de 1500 iteraciones. 

Al realizar la simulación, a partir de los datos en condiciones_iniciales.txt